# Stage 1 — Oracle Pipeline (Colab)

Reproduces the paper's oracle pipeline: CLIP ViT-B/32 embeddings → PCA (or LDA) compression fit on `m` sampled images → oracle compressed embeddings `e' = (e - mean) @ W`.

This is **Stage 1 only** (see `distillation_experiment_prompt.md`) — it does not train the student (Stage 2) or run the full comparison eval (Stage 3). It produces everything Stage 2 needs: cached CLIP embeddings, the fitted `W`/`mean`, and the oracle targets for the fit and eval sets.

Run this notebook on a Colab GPU runtime (Runtime → Change runtime type → GPU).

In [2]:
import os

REPO_URL = "https://github.com/Nizaxga/distillation-embedding-experiment.git"
REPO_DIR = "/content/distillation-embedding-experiment"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}


Cloning into '/content/distillation-embedding-experiment'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 35 (delta 0), reused 35 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 29.51 KiB | 530.00 KiB/s, done.
/content/distillation-embedding-experiment


In [3]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00


## Optional: mount Drive

Two independent reasons to mount Drive (see `docs/SETUP.md`):
1. **Data** — reuse a precached `imagenet-dog-15/<label>/*.jpg` dir if one already exists (shared with `colab_transfer_disentangler.ipynb`), instead of the slower gated-HF fallback.
2. **Persistence** — survive a Colab session reset without manually re-downloading/re-extracting.

Skip this cell if you don't have/want Drive access — the notebook falls back to the HF `imagenet-1k` streaming path automatically (requires `huggingface-cli login`, see next cell).

In [4]:
from google.colab import drive

drive.mount("/content/drive")

# Same save_dir convention as colab_transfer_disentangler.ipynb / colab_decoder_latent_visualizer.ipynb.
save_dir = "/content/drive/MyDrive/representation-learning"
print(f"[LOG] save_dir={save_dir}")

Mounted at /content/drive
[LOG] save_dir=/content/drive/MyDrive/representation-learning


In [5]:
import os

# Same save_dir/.cache/<name>/<label>/*.jpg layout used by colab_transfer_disentangler.ipynb
# and colab_decoder_latent_visualizer.ipynb (CATS_DIR = save_dir/.cache/imagenet-transfer/cats) --
# reuses a precached dir instead of re-downloading through the gated HF `imagenet-1k` dataset.
os.environ["IMAGENET_DOG15_CACHE"] = os.path.join(save_dir, ".cache/imagenet-dog-15")

# Alternative: point at a full local ImageNet-1k ImageFolder instead (root/<synset>/*.JPEG).
# os.environ["IMAGENET_ROOT"] = "/content/drive/MyDrive/imagenet/train"

# If neither is set/found, load_dog15() falls back to the gated HF `imagenet-1k` stream --
# uncomment and run once if you need that path:
# !huggingface-cli login

In [6]:
# Optional: symlink outputs/ onto Drive so embedding cache + compression artifacts survive
# a session reset -- mirrors save_dir/output-transfer-disentangler/... in the sibling
# notebooks. Skip if you'll sync outputs/ down manually instead (docs/SETUP.md).
DRIVE_OUTPUTS = os.path.join(save_dir, "output-distillation-embedding-experiment")

if os.path.isdir("/content/drive/MyDrive") and not os.path.exists("outputs"):
    os.makedirs(DRIVE_OUTPUTS, exist_ok=True)
    os.symlink(DRIVE_OUTPUTS, "outputs")

In [7]:
import sys

sys.path.insert(0, REPO_DIR)

import numpy as np

from src.compress.fit import fit_pca, fit_lda, save_compression, transform
from src.data.imagenet_dog15 import load_dog15, split_fit_eval
from src.device import get_device
from src.embeddings.cache import load_embeddings
from src.embeddings.extract import extract_clip_embeddings
from src.utils.config import load_config

In [10]:
cfg = load_config("configs/pilot_dog15_clip_pca.yaml")
device = get_device()
print(f"run_name={cfg.run_name}  method={cfg.method}  k={cfg.k}  m={cfg.m}")

[device] using CUDA: Tesla T4
run_name=imagenet_dog15_clip_vit_b32_pca_k32_m1000_small_cnn_seed0  method=pca  k=32  m=1000


## 1. Load data + deterministic fit/eval split

In [11]:
records = load_dog15(seed=cfg.seed)
fit_records, eval_records = split_fit_eval(records, cfg.m, seed=cfg.seed)
print(f"total={len(records)}  fit={len(fit_records)}  eval={len(eval_records)}")

[data] label dirs under /content/drive/MyDrive/representation-learning/.cache/imagenet-dog-15 aren't synset ids (['0', '1', '10', '11', '12', '13', '14', '2', '3', '4', '5', '6', '7', '8', '9']); assigning class indices by sort order.
[data] loaded 1500 ImageNet-Dog-15 images from precached dir /content/drive/MyDrive/representation-learning/.cache/imagenet-dog-15
total=1500  fit=1000  eval=500


## 2. Extract CLIP ViT-B/32 embeddings

HEAVY (one CLIP forward pass per image) but disk-cached by image id under `outputs/embeddings/`. Safe to rerun — already-cached images are skipped.

In [12]:
extract_clip_embeddings(records, cfg.dataset, cfg.backbone, device=device)

[extract] 1500/1500 embeddings to compute on cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
extracting: 100%|██████████| 24/24 [11:09<00:00, 27.92s/it]


## 3. Fit compression on the fit set only

`W` (and `mean`) are fit **only** on the `m` fit-set embeddings — matching the paper's motivating scenario where the local encoder never sees more foundation-model access than this. `eval_records` are held out entirely from this step.

In [13]:
fit_ids = [r.image_id for r in fit_records]
eval_ids = [r.image_id for r in eval_records]
fit_full = load_embeddings(cfg.dataset, cfg.backbone, fit_ids)
eval_full = load_embeddings(cfg.dataset, cfg.backbone, eval_ids)

mean = fit_full.mean(axis=0)
if cfg.method == "pca":
    W, _ = fit_pca(fit_full - mean, cfg.k, seed=cfg.seed)
elif cfg.method == "lda":
    fit_labels = np.array([r.label for r in fit_records])
    W, _ = fit_lda(fit_full - mean, fit_labels, cfg.k)
else:
    raise ValueError(cfg.method)

save_compression(cfg.run_name, W, mean, cfg.method)
print(f"W shape={W.shape}  saved to outputs/compression/{cfg.run_name}/")

W shape=(512, 32)  saved to outputs/compression/imagenet_dog15_clip_vit_b32_pca_k32_m1000_small_cnn_seed0/


In [14]:
# e' for the m fit images -> Stage 2's student regression target.
fit_targets = transform(fit_full, W, mean)
# oracle embeddings for the held-out eval set -> Stage 3's upper-bound row.
eval_oracle = transform(eval_full, W, mean)

np.save(f"outputs/compression/{cfg.run_name}/fit_targets.npy", fit_targets)
np.save(f"outputs/compression/{cfg.run_name}/eval_oracle.npy", eval_oracle)
print(f"fit_targets={fit_targets.shape}  eval_oracle={eval_oracle.shape}")

fit_targets=(1000, 32)  eval_oracle=(500, 32)


## 4. Sanity check — oracle clustering score

No student involved yet — this just confirms `W` is correct before building Stage 2/3 on top of it. Compare the `oracle_<method>` row below against the paper's Table I (CLIP) ImageNet-Dog-15/PCA number. If it doesn't roughly match, something in data loading, embedding extraction, or PCA fitting is off — fix that before training a student.

In [15]:
from src.eval.clustering import cluster_and_score

eval_labels = np.array([r.label for r in eval_records])
n_clusters = len(set(r.label for r in records))

full_scores = cluster_and_score(eval_full, eval_labels, n_clusters, cfg.eval_seeds)
oracle_scores = cluster_and_score(eval_oracle, eval_labels, n_clusters, cfg.eval_seeds)

metrics = ["v_measure", "nmi", "ari", "acc"]
print(f"{'row':<14}" + "".join(f"{m:>12}" for m in metrics))
for name, s in [("full", full_scores), (f"oracle_{cfg.method}", oracle_scores)]:
    print(f"{name:<14}" + "".join(f"{s[m]:>12.4f}" for m in metrics))

row              v_measure         nmi         ari         acc
full                0.6060      0.6060      0.4090      0.5840
oracle_pca          0.6491      0.6491      0.4833      0.6520


## Done

`outputs/embeddings/` and `outputs/compression/<run_name>/` (`W.npy`, `mean.npy`, `fit_targets.npy`, `eval_oracle.npy`) now hold everything Stage 2 (student training) needs. If you symlinked `outputs/` onto Drive, this survives a session reset; otherwise zip/download `outputs/` before disconnecting (see `docs/RUNNING.md`).